In [ ]:
import random

from pathlib import Path

from glob import glob
from rich.pretty import pprint

from pydantic_ai import BinaryContent

from common.cache import RedisCache
from por.llm_agents import ImageCaptionGenerator

In [ ]:
image_paths = glob("/resources/images/malika-favre-jpg/*.jpg")
print(f"image_paths: {len(image_paths)}")

In [ ]:
def get_binary_content(image_path: str) -> BinaryContent:
    with open(image_path, "rb") as image_file:
        return BinaryContent(
            data=image_file.read(),
            media_type="image/png",
        )

In [ ]:
prompt = "Provide the a direct, ready-to-use image-generation caption"
user_prompts = [prompt for _ in image_paths]
user_contents = [
    get_binary_content(image_path=image_path) for image_path in image_paths
]

In [ ]:
icg = ImageCaptionGenerator(cache=RedisCache())
icg_outputs = await icg.batch_generate(
    user_prompts=user_prompts,
    user_contents=user_contents,  # type: ignore
)

In [ ]:
icg_output = random.choice(icg_outputs)
pprint(icg_output)

In [ ]:
for image_path, icg_output in zip(image_paths, icg_outputs):
    file_name = Path(image_path).stem
    with open(f"/resources/images/malika-favre-jpg/{file_name}.txt", "w") as f:
        f.write(icg_output.image_caption)